In [ ]:
import pandas as pd
import plotly.express as px
import itertools
from scipy.stats import ttest_ind, pointbiserialr

df = pd.read_parquet('../data/processed/telco_churn.parquet')

display(df.head())
display(df.info())
display(df.isnull().sum())

In [ ]:
# Univariate Analysis
# Распределение целевой переменной Churn
total_customers = df['Churn'].count()
churn_yes = (df['Churn'] == 1).sum()
churn_no = (df['Churn'] == 0).sum()

fig = px.pie(df, names='Churn', title='Распределение оттока клиентов (Churn)',
             color_discrete_sequence=px.colors.qualitative.Set2,
             hole=0.4)
fig.update_traces(textinfo='percent+label')
fig.update_layout(height=500)

print(f"Всего клиентов: {total_customers}")
print(f"Оставшихся (Churn = 1): {churn_no} - {round(churn_no / total_customers * 100.0, 2)}%") 
print(f"Ушедших (Churn = 0): {churn_yes} - {round(churn_yes / total_customers * 100.0, 2)}%")
fig.show() 

In [ ]:
# Общее распределение числовых переменных

numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

for col in numerical_cols:
    fig = px.histogram(df, x=col,
                        marginal='box',
                        nbins=50,
                        title=f'Общее распределение {col}',
                        color_discrete_sequence=['blue'],
                        barmode='overlay')
    fig.update_layout(height=500)

    print(f'Описание {col}:')
    display(df[col].describe())
    fig.show()

In [ ]:
# Таблицы и графики частот для категориальных признаков

categorical_cols = [col for col in df.columns if col not in numerical_cols 
                    and col not in ['customerID', 'Churn', 'customer_type', 
                                    'avg_monthly_per_tenure_month', 'fiber_optic_flag', 
                                    'electronic_check_flag', 'month_to_month_flag']]

for col in categorical_cols:
    freq = df[col].value_counts().head(10)
    percent = df[col].value_counts(normalize=True).head(10) * 100
    table = pd.concat([freq, percent.round(2)], axis=1, keys=['Count', 'Percent (%)'])

    fig = px.pie(df, names=col, title=f'Распределение {col}',
             color_discrete_sequence=px.colors.qualitative.Set2,
             hole=0.4)
    fig.update_traces(textinfo='percent+label')
    fig.update_layout(height=500)

    print(f'Распределение {col}:')
    display(table)
    fig.show()

In [ ]:
# Bivariate Analysis
# Распределение числовых переменных с разделением по Churn

for col in numerical_cols:
    fig = px.histogram(df, x=col, color='Churn', 
                        marginal='box',
                        nbins=50,
                        title=f'Распределение {col} по Churn',
                        color_discrete_sequence=px.colors.qualitative.Set2,
                        barmode='overlay')
    fig.update_layout(height=500)

    print(f'Описание {col} по Churn:')
    display(df.groupby('Churn')[col].describe())
    fig.show()

In [ ]:
# Статистические тесты: влияние числовых признаков на Churn

results = []

for col in numerical_cols:
    group_yes = df[df['Churn'] == 1][col]
    group_no  = df[df['Churn'] == 0][col]
    
    # t-тест
    t_stat, p_value_t = ttest_ind(group_yes, group_no, equal_var=False, nan_policy='omit')
    
    # Point-Biserial Correlation
    corr, p_value_pb = pointbiserialr(df['Churn'], df[col])
    
    # Интерпретация силы связи
    if abs(corr) < 0.1:
        strength = "очень слабая"
    elif abs(corr) < 0.3:
        strength = "слабая"
    elif abs(corr) < 0.5:
        strength = "средняя"
    else:
        strength = "сильная"
    
    results.append({
        'Признак': col,
        'Mean (No Churn)': round(group_no.mean(), 2),
        'Mean (Yes Churn)': round(group_yes.mean(), 2),
        'Разница средних': round(group_yes.mean() - group_no.mean(), 2),
        't-статистика': round(t_stat, 4),
        'p-value (t-test)': f"{p_value_t:.6f}",
        'Корреляция (Point-Biserial)': round(corr, 4),
        'Сила связи': strength,
        'p-value (корр.)': f"{p_value_pb:.6f}"
    })

# Вывод результатов в виде таблицы
print("Статистические тесты влияния числовых признаков на отток")
stats_df = pd.DataFrame(results)
display(stats_df)

In [ ]:
# Анализ оттока (Churn = 1) по категориям

results = []

for col in categorical_cols:
    is_numeric = False
    if pd.api.types.is_numeric_dtype(df[col]):
        is_numeric = True
        df[col] = df[col].astype('category')

    churn_rate = (df.groupby(col, observed=False)['Churn']
                    .mean()
                    .mul(100)           
                    .round(2)
                    .reset_index(name='Churn_Rate'))
    
    churn_rate = churn_rate.sort_values('Churn_Rate', ascending=False)
    churn_rate_display = churn_rate[[col, 'Churn_Rate']]
    
    fig = px.bar(
        churn_rate,
        x=col,
        y='Churn_Rate',
        title=f'Процент оттока (Churn = 1) по {col}',
        color='Churn_Rate',                    
        color_continuous_scale='RdYlGn_r',       
        text='Churn_Rate',
        labels={'Churn_Rate': 'Процент ушедших (%)', col: col}        
    )
    
    fig.update_traces(texttemplate='%{y:.2f}%')
    
    print(f"\nChurn Rate по {col}:")
    display(churn_rate_display.style.hide(axis="index"))
    fig.show()

    for _, row in churn_rate_display.iterrows():
            results.append({
                'category': col,
                'value': f"{row[col]}",
                'churn_rate': row['Churn_Rate']
            })

    if is_numeric:
        df[col] = df[col].astype('int8')

top_category = pd.DataFrame(results)
top10 = top_category.sort_values('churn_rate', ascending=False).head(10).reset_index(drop=True)

display(top10[['category', 'value', 'churn_rate']].style.format({'churn_rate': '{:.2f}%'})
        .set_caption("ТОП-10 самых рискованных категориальных признаков"))    


In [93]:
# Multivariate Analysis
# Анализ комбинаций категориальных признаков

combinations = list(itertools.combinations(['tenure_group', 'PaymentMethod', 'Contract', 
                                            'InternetService', 'OnlineSecurity', 'SeniorCitizen', 
                                            'TechSupport', 'OnlineBackup', 'DeviceProtection'], 2))
results = []

for col1, col2 in combinations:
    rate = (df.groupby([col1, col2], observed=False)['Churn']
        .mean()                 
        .mul(100)                  
        .round(2)
        .reset_index(name='Churn_Rate'))
    
    rate = rate.sort_values('Churn_Rate', ascending=False)
    for _, row in rate.iterrows():
        results.append({
            'col1' : col1,
            'col2' : col2,
            'feature_pair' : f"{col1} x {col2}",
            'combination': f"{row[col1]} x {row[col2]}",
            'churn_rate': row['Churn_Rate']
        })

top_combinations = pd.DataFrame(results)
top10 = top_combinations.sort_values('churn_rate', ascending=False).head(10).reset_index(drop=True)
display(top10[['feature_pair', 'combination', 'churn_rate']].style.format({'churn_rate': '{:.2f}%'})
        .set_caption("ТОП-10 самых рискованных комбинаций (с участием tenure_group)"))


# Исключаем tenure_group из анализа комбинаций
combinations2 = list(itertools.combinations(['PaymentMethod', 'Contract', 
                                            'InternetService', 'OnlineSecurity', 'SeniorCitizen', 
                                            'TechSupport', 'OnlineBackup', 'DeviceProtection'], 2))
results2 = []

for col1, col2 in combinations2:
    
    rate = (df.groupby([col1, col2], observed=False)['Churn']
            .mean()
            .mul(100)
            .round(2)
            .reset_index(name='Churn_Rate'))
    
    for _, row in rate.iterrows():
        results2.append({
            'combination': f"{row[col1]} + {row[col2]}",
            'feature_pair': f"{col1} × {col2}",
            'churn_rate': row['Churn_Rate']
        })

top_combinations2 = pd.DataFrame(results2)
top10_2 = top_combinations2.sort_values('churn_rate', ascending=False).head(10).reset_index(drop=True)

display(top10_2[['feature_pair', 'combination', 'churn_rate']]
        .style.format({'churn_rate': '{:.2f}%'})
        .set_caption("ТОП-10 рискованных комбинаций (исключая tenure_group)"))

,feature_pair,combination,churn_rate
0,tenure_group x InternetService,1-6 months x Fiber optic,74.19%
1,tenure_group x SeniorCitizen,1-6 months x 1,72.81%
2,tenure_group x PaymentMethod,1-6 months x Electronic check,67.20%
3,tenure_group x OnlineSecurity,1-6 months x No,66.23%
4,tenure_group x DeviceProtection,1-6 months x Yes,66.10%
5,tenure_group x TechSupport,1-6 months x No,65.63%
6,tenure_group x OnlineBackup,1-6 months x No,64.26%
7,tenure_group x DeviceProtection,1-6 months x No,62.13%
8,tenure_group x InternetService,7-12 months x Fiber optic,61.06%
9,tenure_group x SeniorCitizen,7-12 months x 1,57.28%


,feature_pair,combination,churn_rate
0,Contract × SeniorCitizen,Month-to-month + 1,54.65%
1,Contract × InternetService,Month-to-month + Fiber optic,54.61%
2,PaymentMethod × OnlineBackup,Electronic check + No,53.92%
3,PaymentMethod × Contract,Electronic check + Month-to-month,53.73%
4,PaymentMethod × SeniorCitizen,Electronic check + 1,53.37%
5,PaymentMethod × InternetService,Electronic check + Fiber optic,53.23%
6,PaymentMethod × TechSupport,Electronic check + No,53.18%
7,PaymentMethod × OnlineSecurity,Electronic check + No,53.17%
8,SeniorCitizen × OnlineBackup,1 + No,52.77%
9,PaymentMethod × DeviceProtection,Electronic check + No,52.06%


### **Key Insights & Business Recommendations**

Ключевые выводы:

#### 1. **Общий уровень оттока**
Общий Churn Rate составляет **26.54%** (ушли 1869 клиентов из 7043).


#### 2. Самые сильные факторы оттока

**Топ-5 наиболее значимых драйверов оттока:**

1. **Способ оплаты (PaymentMethod)**  
   Самый высокий отток наблюдается у клиентов с методом оплаты `Electronic check` — **XX.XX%**.

2. **Тип контракта (Contract)**  
   Клиенты с помесячным контрактом (`Month-to-month`) уходят в **XX.XX%** случаев, что значительно выше, чем у контрактов на 1 год (**XX.XX%**) и 2 года (**XX.XX%**).

3. **Длительность пользования (tenure)**  
   Клиенты, которые пользуются услугами менее 6 месяцев, имеют churn rate **XX.XX%**. С увеличением стажа отток существенно снижается.

4. **Тип интернет-соединения (InternetService)**  
   Пользователи `Fiber optic` демонстрируют высокий отток — **XX.XX%**, в то время как клиенты без интернета уходят всего в **XX.XX%** случаев.

5. **Дополнительные услуги**  
   Отсутствие `TechSupport` приводит к оттоку в **XX.XX%** случаев, `OnlineSecurity` — **XX.XX%**. Наличие этих услуг заметно снижает риск ухода.

#### 3. Демографические и поведенческие особенности
- Пожилые клиенты (`SeniorCitizen = Yes`) уходят значительно чаще — **XX.XX%**.
- Клиенты с бумажным биллингом (`PaperlessBilling = No`) показывают более низкий отток (**XX.XX%** против **XX.XX%**).

#### 4. Слабые факторы
- Пол клиента (`gender`) практически не влияет на отток (разница менее XX%).
- Наличие/отсутствие `PhoneService` имеет минимальное влияние.

#### 5. Основные гипотезы
- Основной причиной оттока является **неудовлетворённость ценой и гибкостью контракта**.
- Клиенты, использующие дорогие и современные услуги (Fiber optic), ожидают высокого качества, которого, возможно, не получают.
- Отсутствие дополнительных услуг (особенно технической поддержки) снижает лояльность клиентов.

### Рекомендации для бизнеса (предварительные)
- Стимулировать переход клиентов с помесячного контракта на годовые/двухгодовые.
- Пересмотреть условия оплаты через Electronic check или предложить бонусы за другие способы оплаты.
- Улучшить качество поддержки (`TechSupport`) и киберзащиты (`OnlineSecurity`).
- Разработать специальные программы удержания для новых клиентов (первые 6 месяцев).